# CplusyAIT GPTQ Ternary Quantization
Run on **Colab GPU runtime** (T4+). Do NOT run locally.

In [ ]:
!pip install -q torch transformers datasets psutil numpy --index-url https://download.pytorch.org/whl/cu121

In [16]:
import torch, psutil
ram = psutil.virtual_memory().total / (1024**3)
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'
print(f'RAM: {ram:.1f} GB | GPU: {gpu}')
assert ram > 10, 'Use Colab runtime, not local!'

RAM: 12.7 GB | GPU: Tesla T4


In [17]:
import os
if os.path.isdir('CplusyAIT'):
    !cd CplusyAIT && git pull
else:
    !git clone https://github.com/OCDDEVS/CplusyAIT.git
!ls CplusyAIT/scripts/

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 1), reused 3 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 4.79 KiB | 4.79 MiB/s, done.
From https://github.com/OCDDEVS/CplusyAIT
   8287465..8821872  main       -> origin/main
Updating 8287465..8821872
Fast-forward
 demo.ipynb | 316 +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++--
 1 file changed, 309 insertions(+), 7 deletions(-)
download_model.py  quantize_gptq.py


In [ ]:
# Quantize TinyLlama 1.1B to ternary
!python CplusyAIT/scripts/quantize_gptq.py \
  --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --output CplusyAIT/models/TinyLlama-1.1B-GPTQ-ternary \
  --device cuda \
  --mem-limit 11

In [ ]:
# Check output
import os, json
out = 'CplusyAIT/models/TinyLlama-1.1B-GPTQ-ternary'
manifest = json.load(open(os.path.join(out, 'manifest.json')))
wb = os.path.getsize(os.path.join(out, 'weights.bin')) / (1024**2)
print(f'Tensors: {len(manifest["tensors"])}')
print(f'Weights: {wb:.1f} MB')

# Verify per-channel gammas are present
gamma_tensors = [t for t in manifest['tensors'] if '__gamma__' in t['name']]
print(f'Per-channel gamma tensors: {len(gamma_tensors)}')
assert len(gamma_tensors) > 0, 'GPTQ quantizer should produce per-channel gamma tensors'

## Run inference on Colab
Install Rust toolchain, compile the project, and run the ternary inference engine.

In [ ]:
# Install Rust (skip if already installed)
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ['PATH']}"
!rustc --version

In [ ]:
# Build the infer binary (~3-5 min first time)
!cd CplusyAIT && cargo build --release --bin infer 2>&1 | tail -5

## Test 1: Greedy decoding (temperature=0)
Deterministic output — the strongest signal for whether the model is working.

In [ ]:
!cd CplusyAIT && ./target/release/infer \
  --model models/TinyLlama-1.1B-GPTQ-ternary \
  --prompt "The meaning of life is" \
  --max-tokens 50 \
  --temperature 0

## Test 2: Sampled generation (temperature=0.7)
With top-k sampling — should produce varied but coherent text.

In [ ]:
!cd CplusyAIT && ./target/release/infer \
  --model models/TinyLlama-1.1B-GPTQ-ternary \
  --prompt "The meaning of life is" \
  --max-tokens 50 \
  --temperature 0.7

## Test 3: Code generation prompt

In [ ]:
!cd CplusyAIT && ./target/release/infer \
  --model models/TinyLlama-1.1B-GPTQ-ternary \
  --prompt "def fibonacci(n):" \
  --max-tokens 80 \
  --temperature 0

## Test 4: Chat-style prompt
TinyLlama-Chat was fine-tuned for chat — test with a chat template.

In [ ]:
!cd CplusyAIT && ./target/release/infer \
  --model models/TinyLlama-1.1B-GPTQ-ternary \
  --prompt "<|user|>\nWhat is the capital of France?\n<|assistant|>\n" \
  --max-tokens 40 \
  --temperature 0

## Test 5: Longer generation with low temperature

In [ ]:
!cd CplusyAIT && ./target/release/infer \
  --model models/TinyLlama-1.1B-GPTQ-ternary \
  --prompt "Once upon a time in a land far away," \
  --max-tokens 100 \
  --temperature 0.3

In [ ]:
# Download the quantized model to your machine
from google.colab import files
!cd CplusyAIT/models && zip -r /tmp/TinyLlama-ternary.zip TinyLlama-1.1B-GPTQ-ternary/
files.download('/tmp/TinyLlama-ternary.zip')